# Phase 1 sweep — Kaggle

Kaggle suits this job better than Colab: **4 CPU cores** instead of 2, and
**Save & Run All** executes the notebook in batch with no browser tab open, so
nothing dies when you close your laptop.

**Do not enable the GPU accelerator.** These networks are 128×128 at batch 64, so
kernel-launch overhead dominates and a GPU is typically no faster — sometimes
slower. Leave the accelerator on *None* and keep your GPU quota.

**Settings to check before running:** Accelerator *None*, Internet *On*
(needed for pip and the git clone), Persistence *Files only*.

Everything is resumable, so re-running this notebook continues rather than
restarts. See `experiments/README.md` for the design and options.

## 1. Setup

In [ ]:
!pip install -q 'gymnasium[box2d]' swig

import os, multiprocessing
print('cores:', multiprocessing.cpu_count())

In [ ]:
REPO_URL = 'https://github.com/azkaanasir/Transfer-Learning-in-Double-and-Dueling-DQNs.git'
BRANCH   = 'phase0-audit-phase1-runner'
REPO     = '/kaggle/working/repo'

# /kaggle/working persists as notebook output; /kaggle/temp does not.
OUT_ROOT = '/kaggle/working/runs'

import os
if not os.path.exists(REPO):
    !git clone -q --branch $BRANCH $REPO_URL $REPO
os.makedirs(OUT_ROOT, exist_ok=True)
%cd $REPO
!git log --oneline -1

## 2. Preflight

Verifies both environments build — Box2D failures break LunarLander but not
CartPole, so this is the check worth having — and estimates runtime for this
machine.

In [ ]:
!python experiments/preflight.py

## 3. Pilot — 2 seeds

Gives real timings on this machine and, more importantly, confirms the CartPole
sources actually learn. The published DDQN source reached 26.94 on a task solved
at 195; check the CartPole rows below are far above that before scaling up.

In [ ]:
!python experiments/sweep.py --seeds 0 1 --stage all --jobs 2 --out-root $OUT_ROOT

In [ ]:
!python experiments/aggregate.py --out-root $OUT_ROOT

## 4. Full sweep

A 12-hour session may not finish all 120 runs. That is fine — re-run this cell in
a new session and it resumes from the checkpoints, provided Persistence is set to
*Files only* so `/kaggle/working` survives.

In [ ]:
!python experiments/sweep.py --seeds 0-9 --stage all --jobs 4 --out-root $OUT_ROOT

In [ ]:
# Progress check — safe to run any time, including against a partial sweep.
!python experiments/sweep.py --seeds 0-9 --stage all --out-root $OUT_ROOT --dry-run | head -5

## 5. Results

In [ ]:
!python experiments/aggregate.py --out-root $OUT_ROOT

In [ ]:
!python experiments/stats.py --per-seed $OUT_ROOT/per_seed.csv

In [ ]:
# per_seed.csv is the artifact every number in the paper should trace to.
# Copy it to the notebook output root so it is easy to download.
!cp $OUT_ROOT/per_seed.csv /kaggle/working/per_seed.csv

import pandas as pd
df = pd.read_csv('/kaggle/working/per_seed.csv')
print(df.groupby(['env_id', 'arm']).size())
df.head()